<a href="https://colab.research.google.com/github/aayushrouniyarcs24-cmd/AI-Lab-1BF24CS010/blob/main/1BF24CS010_Week6_FOL_Unification_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [9]:
import ast

print("Enter expressions as Python lists (e.g., ['f', 'x', 'A'] where lowercase single letters are variables):")
try:
    # Using input() so you can dynamically test different expressions
    expr_input1 = input("Enter Expression 1: ")
    expr_input2 = input("Enter Expression 2: ")

    # Parse the string input safely into list objects
    user_expr1 = ast.literal_eval(expr_input1)
    user_expr2 = ast.literal_eval(expr_input2)

    if not isinstance(user_expr1, list) or not isinstance(user_expr2, list):
        raise ValueError("Expressions must be provided in list format.")

    print(f"\nAttempting to unify {user_expr1} and {user_expr2}...")
    result = unify(user_expr1, user_expr2)

    if result is not None:
        print("Success! Substitution:", result)
    else:
        print("Failure: The expressions do not unify.")
except Exception as e:
    print(f"Error parsing input: {e}. Please ensure you enter valid list representations, e.g., ['f', 'x', 'A']")

Enter expressions as Python lists (e.g., ['f', 'x', 'A'] where lowercase single letters are variables):
Enter Expression 1: ['Eats','x','Apple']
Enter Expression 2: ['Eats','Riya','y']

Attempting to unify ['Eats', 'x', 'Apple'] and ['Eats', 'Riya', 'y']...
Success! Substitution: {'x': 'Riya', 'y': 'Apple'}


In [ ]:
def is_variable(x):
    return isinstance(x, str) and x.islower() and len(x) == 1

def occurs_check(var, term):
    if var == term:
        return True
    if isinstance(term, list):
        return any(occurs_check(var, t) for t in term)
    return False

def apply_subst(subst, expr):
    if not subst:
        return expr
    if isinstance(expr, str):
        return subst.get(expr, expr)
    if isinstance(expr, list):
        return [apply_subst(subst, arg) for arg in expr]
    return expr

def parse_expression(text):
    text = text.replace(" ", "")
    if '(' not in text:
        return text

    idx = text.find('(')
    name = text[:idx]
    args_str = text[idx+1:-1] if text.endswith(')') else text[idx+1:]

    args = []
    current = []
    paren_depth = 0

    for char in args_str:
        if char == ',' and paren_depth == 0:
            args.append(parse_expression("".join(current)))
            current = []
        else:
            if char == '(':
                paren_depth += 1
            elif char == ')':
                paren_depth -= 1
            current.append(char)

    if current:
        args.append(parse_expression("".join(current)))

    return [name] + args

def format_expression(expr):
    if isinstance(expr, list):
        if len(expr) == 1:
            return f"{expr}()"
        return f"{expr}({', '.join(format_expression(e) for e in expr[1:])})"
    return str(expr)

def unify(psi1, psi2, subst=None):
    if subst is None:
        subst = {}

    if isinstance(psi1, str) and isinstance(psi2, str):
        if psi1 == psi2:
            return subst
        if is_variable(psi1):
            if occurs_check(psi1, psi2): return "FAILURE"
            subst[psi1] = psi2
            return subst
        if is_variable(psi2):
            if occurs_check(psi2, psi1): return "FAILURE"
            subst[psi2] = psi1
            return subst
        return "FAILURE"

    if is_variable(psi1):
        if occurs_check(psi1, psi2): return "FAILURE"
        subst[psi1] = psi2
        return subst
    if is_variable(psi2):
        if occurs_check(psi2, psi1): return "FAILURE"
        subst[psi2] = psi1
        return subst

    if isinstance(psi1, list) and isinstance(psi2, list):
        if psi1[0] != psi2[0]:
            return "FAILURE"
        if len(psi1) != len(psi2):
            return "FAILURE"

        for i in range(1, len(psi1)):
            arg1 = apply_subst(subst, psi1[i])
            arg2 = apply_subst(subst, psi2[i])

            S = unify(arg1, arg2, subst)
            if S == "FAILURE":
                return "FAILURE"
            subst.update(S)

        return subst

    return "FAILURE"



while True:
    input_A = input("Enter Expression A (or 'exit' to quit): ")
    if input_A.lower() == 'exit':
        break

    input_B = input("Enter Expression B: ")

    expr_A = parse_expression(input_A)
    expr_B = parse_expression(input_B)

    print("\n[Parsed Structure A]:", expr_A)
    print("[Parsed Structure B]:", expr_B)

    result = unify(expr_A, expr_B, subst={})

    print("\n--- Result ---")
    if result == "FAILURE":
        print("Unification Failed!")
    else:
        print("Success! Unification  applied:")
        if not result:
            print("  None")
        for var, val in result.items():
            print(f"  {var} -> {format_expression(val)}")

        unified_output = apply_subst(result, expr_A)
        print(f"\nUnified Form: {format_expression(unified_output)}")
    print("-" * 50 + "\n")






Enter Expression A (or 'exit' to quit): Eats(x,Apple)
Enter Expression B: Eats(Riya,Y)

[Parsed Structure A]: ['Eats', 'x', 'Apple']
[Parsed Structure B]: ['Eats', 'Riya', 'Y']

--- Result ---
Unification Failed!
--------------------------------------------------

Enter Expression A (or 'exit' to quit): Eats(x,Apple)
Enter Expression B: Eats(Riya,y)

[Parsed Structure A]: ['Eats', 'x', 'Apple']
[Parsed Structure B]: ['Eats', 'Riya', 'y']

--- Result ---
Success! Substitutions applied:
  x -> Riya
  y -> Apple

Unified Form: ['Eats', 'Riya', 'Apple'](Riya, Apple)
--------------------------------------------------

Enter Expression A (or 'exit' to quit): Eats(Apple)
Enter Expression B: Eats(Riya,y)

[Parsed Structure A]: ['Eats', 'Apple']
[Parsed Structure B]: ['Eats', 'Riya', 'y']

--- Result ---
Unification Failed!
--------------------------------------------------

Enter Expression A (or 'exit' to quit): Eats(x,Apple)
Enter Expression B: Likes(Riya,y)

[Parsed Structure A]: ['Eats', '